# Ensemble Methods

## Overview

This notebook demonstrates the public classifier builders in ensemble_methods using a synthetic binary classification dataset.
- Problem: correlated base estimators limit the variance reduction available from bagging.
- Approach: fit bagging, random-forest, and boosting classifiers, then inspect the bagging variance relationship.
- Build And Fit Ensemble Classifiers: It compares held-out classification metrics.
- Bagging Prediction Variance: It plots the relationship in AFML Figure 6.1.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


In [2]:
from src.strategy_modeling.ensemble_methods import (
    build_bagging_classifier,
    build_boosting_classifier,
    build_random_forest_classifier,
)
from src.strategy_modeling.feature_importance import get_test_data

## Load And Split Classification Data

This cell creates a reproducible classification dataset and stratified train/test split.
- Informative, redundant, and noise columns retain their known synthetic roles.
- Sample weights exercise the weighted estimator fitting path.


In [3]:
trns_x, cont = get_test_data(
    n_features=12,
    n_informative=4,
    n_redundant=3,
    n_samples=360,
    random_state=21,
)
y = cont["bin"]
sample_weight = cont["w"]

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    trns_x,
    y,
    sample_weight,
    test_size=0.30,
    shuffle=True,
    stratify=y,
    random_state=21,
)

pd.Series(
    {
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "num_features": trns_x.shape[1],
        "train_positive_rate": y_train.mean(),
        "test_positive_rate": y_test.mean(),
    }
)

train_rows             252.000000
test_rows              108.000000
num_features            12.000000
train_positive_rate      0.496032
test_positive_rate       0.490741
dtype: float64

## Build And Fit Ensemble Classifiers

This cell fits bagging, random-forest, and AdaBoost classifiers and reports held-out metrics.
- The three models use the source-module builders.
- Accuracy and F1 score compare predictions on the same held-out sample.


In [4]:
models = {
    "Bagging": build_bagging_classifier(
        n_estimators=50,
        max_samples=0.75,
        n_jobs=1,
        random_state=21,
    ),
    "Random Forest": build_random_forest_classifier(
        n_estimators=50,
        n_jobs=1,
        random_state=21,
    ),
    "AdaBoost": build_boosting_classifier(
        n_estimators=50,
        learning_rate=0.5,
        max_depth=1,
        random_state=21,
    ),
}

metric_rows = []
fitted_models = {}
for name, clf in models.items():
    fit = clf.fit(X_train, y_train, sample_weight=w_train.values)
    pred = fit.predict(X_test)
    fitted_models[name] = fit
    metric_rows.append(
        {
            "models": name,
            "accuracy": accuracy_score(y_test, pred),
            "f1": f1_score(y_test, pred),
            "train_score": fit.score(X_train, y_train),
            "test_score": fit.score(X_test, y_test),
        }
    )

metrics = pd.DataFrame(metric_rows).set_index("models")
metrics

,accuracy,f1,train_score,test_score
model,,,,
Bagging,0.759259,0.763636,0.988095,0.759259
Random Forest,0.768519,0.770642,1.000000,0.768519
AdaBoost,0.777778,0.777778,0.928571,0.777778


## Bagging Prediction Variance

This cell plots the standard deviation of a bagged prediction.
- The curves follow AFML Figure 6.1 for different average correlations between base estimators.
- Lower correlation allows additional estimators to reduce prediction variance.


In [ ]:
num_estimators = np.arange(5, 31)
for correlation in [0.0, 0.25, 0.5, 0.75, 1.0]:
    standard_deviation = np.sqrt(correlation + (1 - correlation) / num_estimators)
    plt.plot(num_estimators, standard_deviation, label=f"average correlation = {correlation:.2f}")

plt.title("AFML Figure 6.1: Standard deviation of a bagged prediction")
plt.xlabel("number of estimators")
plt.ylabel("prediction standard deviation")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
